In [10]:

# Dairyland Collegiate League (DCL) — Game Outcome Prediction Model

# Trains and evaluates multiple ML classifiers to predict whether the home
# team wins a given game, based on:
# - Starting pitcher matchup stats
# - Lineup quality aggregates
# - Team season-to-date records
# - Weather and ballpark conditions
# - Derived differential features

import sys
import json
import joblib
import warnings
import numpy as np
import pandas as pd

from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import (
    train_test_split, StratifiedKFold, cross_validate
)
from sklearn.metrics import (
    accuracy_score, roc_auc_score, classification_report,
    confusion_matrix, brier_score_loss, log_loss
)
from sklearn.calibration import CalibratedClassifierCV
from sklearn.inspection import permutation_importance

warnings.filterwarnings("ignore")

# 1. LOAD DATA

def load_data(path="/Users/drewrasmussen/Desktop/Personal/DCL/dcl_games.csv"):
    df = pd.read_csv(path)
    print(f"Loaded {len(df):,} games | {df.columns.size} columns")
    print(f"Seasons: {sorted(df['season'].unique())}")
    print(f"Teams:   {sorted(df['home_team'].unique())}")
    print(f"Home win rate: {df['home_team_wins'].mean():.3f}\n")
    return df

# 2. FEATURE ENGINEERING

WEATHER_MAP = {
    "Clear": 0, "Partly Cloudy": 1, "Overcast": 2,
    "Light Rain": 3, "Heavy Rain": 4, "Windy": 2
}

WIND_DIR_MAP = {
    "Calm": 0, "In": -1, "Out": 1, "L-R": 0, "R-L": 0
}

FEATURE_COLS = [
    # Pitching matchup
    "sp_era_diff",          # positive = home pitcher has ERA advantage
    "sp_whip_diff",         # positive = home pitcher has WHIP advantage
    "sp_k9_diff",           # positive = home pitcher strikes out more
    "sp_bb9_diff",          # positive = home pitcher walks fewer

    "home_sp_era",
    "home_sp_whip",
    "home_sp_k9",
    "home_sp_bb9",
    "home_sp_innings_pitched",
    "home_sp_days_rest",

    "away_sp_era",
    "away_sp_whip",
    "away_sp_k9",
    "away_sp_bb9",
    "away_sp_innings_pitched",
    "away_sp_days_rest",

    # Lineup quality
    "lineup_ops_diff",      # positive = home lineup better
    "lineup_skill_diff",

    "home_lineup_avg_ops",
    "home_lineup_avg_ba",
    "home_lineup_total_hr",
    "home_lineup_avg_krate",
    "home_lineup_avg_bbrate",
    "home_lineup_avg_skill",

    "away_lineup_avg_ops",
    "away_lineup_avg_ba",
    "away_lineup_total_hr",
    "away_lineup_avg_krate",
    "away_lineup_avg_bbrate",
    "away_lineup_avg_skill",

    # Team season-to-date
    "win_pct_diff",
    "run_diff_diff",
    "last10_wins_diff",

    "home_team_win_pct",
    "home_team_run_diff",
    "home_team_last10_wins",
    "home_team_home_win_pct",
    "home_team_runs_for",
    "home_team_runs_against",

    "away_team_win_pct",
    "away_team_run_diff",
    "away_team_last10_wins",
    "away_team_away_win_pct",
    "away_team_runs_for",
    "away_team_runs_against",

    # Ballpark
    "park_factor",

    # Weather
    "weather_encoded",
    "temperature_f",
    "wind_mph",
    "wind_dir_encoded",
    "humidity_pct",
    "precipitation_chance",

    # Situational(Experimental)
    "ump_k_rate_bias",
    "attendance_pct_capacity",
    "game_month",
]

def engineer_features(df):
    df = df.copy()

    # Encode categoricals
    df["weather_encoded"]  = df["weather_condition"].map(WEATHER_MAP).fillna(1).astype(int)
    df["wind_dir_encoded"] = df["wind_direction"].map(WIND_DIR_MAP).fillna(0).astype(int)

    # Park capacity lookup
    park_caps = {
        "River Rats Ballpark": 180,
        "Cheese Kings Ballpark": 240,
        "Bobbers Ballpark": 160,
        "Beach Bums Ballpark": 200,
        "Mapaches Ballpark": 140,
        "Foxes Ballpark": 220,
    }
    df["park_capacity"] = df["ballpark"].map(park_caps).fillna(2000)
    df["attendance_pct_capacity"] = (df["attendance"] / df["park_capacity"]).round(3)

    # Drop early-season games where records are very sparse (< 5 games played)
    df["home_games_played"] = df["home_team_wins"] + df["home_team_losses"]
    # We don't have cumulative at-game-time in this col, so approximate:
    # (already built in as STD records)

    return df



# 3. TRAIN / TEST SPLIT

def split_data(df):

# seasons 1 & 2 for training, season 3 for test.

    train = df[df["season"] <= 2].copy()
    test  = df[df["season"] == 3].copy()

    X_train = train[FEATURE_COLS]
    y_train = train["home_team_wins"]
    X_test  = test[FEATURE_COLS]
    y_test  = test["home_team_wins"]

    print(f"Training set: {len(X_train):,} games (seasons 1–2)")
    print(f"Test set:     {len(X_test):,}  games (season 3)")
    print(f"Train home win rate: {y_train.mean():.3f}")
    print(f"Test  home win rate: {y_test.mean():.3f}\n")

    return X_train, X_test, y_train, y_test, train, test



# 4. MODEL DEFINITIONS


def build_models():
    models = {

        "Logistic Regression": Pipeline([
            ("scaler", StandardScaler()),
            ("clf", LogisticRegression(
                C=0.5, max_iter=2000, solver="lbfgs",
                class_weight="balanced", random_state=42
            )),
        ]),

        "Random Forest": RandomForestClassifier(
            n_estimators=400,
            max_depth=10,
            min_samples_leaf=8,
            min_samples_split=16,
            max_features="sqrt",
            class_weight="balanced",
            oob_score=True,
            n_jobs=-1,
            random_state=42,
        ),

        "Gradient Boosting": GradientBoostingClassifier(
            n_estimators=300,
            learning_rate=0.04,
            max_depth=4,
            min_samples_leaf=12,
            subsample=0.80,
            max_features="sqrt",
            random_state=42,
        ),

    }
    return models

# 5. CROSS-VALIDATION EVALUATION

def cross_validate_models(models, X_train, y_train):
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scoring = ["accuracy", "roc_auc", "neg_log_loss", "neg_brier_score"]

    cv_results = {}
    print("─" * 60)
    print("Cross-Validation Results (5-Fold, Training Set)")
    print("─" * 60)

    for name, model in models.items():
        scores = cross_validate(model, X_train, y_train, cv=cv,
                                scoring=scoring, n_jobs=-1)
        cv_results[name] = {
            "cv_accuracy_mean": round(scores["test_accuracy"].mean(), 4),
            "cv_accuracy_std": round(scores["test_accuracy"].std(),  4),
            "cv_roc_auc_mean": round(scores["test_roc_auc"].mean(),  4),
            "cv_roc_auc_std": round(scores["test_roc_auc"].std(),   4),
            "cv_log_loss_mean": round(-scores["test_neg_log_loss"].mean(), 4),
            "cv_brier_mean": round(-scores["test_neg_brier_score"].mean(), 4),
        }
        r = cv_results[name]
        print(f"\n  {name}")
        print(f"    Accuracy: {r['cv_accuracy_mean']:.4f} ± {r['cv_accuracy_std']:.4f}")
        print(f"    ROC-AUC: {r['cv_roc_auc_mean']:.4f} ± {r['cv_roc_auc_std']:.4f}")
        print(f"    Log-Loss: {r['cv_log_loss_mean']:.4f}")
        print(f"    Brier: {r['cv_brier_mean']:.4f}")

    print()
    return cv_results

# 6. FULL TRAINING + TEST EVALUATION

def train_and_evaluate(models, X_train, X_test, y_train, y_test):
    results = {}
    trained = {}

    print("─" * 60)
    print("Hold-Out Test Set Results (Season 3)")
    print("─" * 60)

    for name, model in models.items():
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        y_prob = model.predict_proba(X_test)[:, 1]

        acc    = accuracy_score(y_test, y_pred)
        auc    = roc_auc_score(y_test, y_prob)
        ll     = log_loss(y_test, y_prob)
        brier  = brier_score_loss(y_test, y_prob)
        cm     = confusion_matrix(y_test, y_pred)
        report = classification_report(y_test, y_pred,
                                       target_names=["Away Win", "Home Win"],
                                       output_dict=True)

        oob = getattr(model, "oob_score_", None)

        results[name] = {
            "accuracy": round(acc, 4),
            "roc_auc": round(auc, 4),
            "log_loss": round(ll,  4),
            "brier": round(brier, 4),
            "confusion_matrix": cm.tolist(),
            "classification_report": report,
            "oob_score": round(oob, 4) if oob else None,
        }
        trained[name] = model

        print(f"\n  {name}")
        print(f" Accuracy: {acc:.4f}")
        print(f" ROC-AUC: {auc:.4f}")
        print(f" Log-Loss: {ll:.4f}")
        print(f" Brier: {brier:.4f}")
        if oob:
            print(f"OOB Score: {oob:.4f}")
        print(f" Confusion Matrix:")
        print(f" [[TN={cm[0,0]:4d}  FP={cm[0,1]:4d}]] (Away wins, predicted as Home win)")
        print(f" [[FN={cm[1,0]:4d}  TP={cm[1,1]:4d}]] (Home wins, predicted correctly)")

    print()
    return results, trained

# 7. SELECTING BEST MODEL & FEATURE IMPORTANCE

def select_best(results, trained):
    # Rank by ROC-AUC
    best_name = max(results, key=lambda k: results[k]["roc_auc"])
    best_model = trained[best_name]
    print(f"★  Best model selected: {best_name}  "
          f"(ROC-AUC={results[best_name]['roc_auc']:.4f})\n")
    return best_name, best_model


def get_feature_importance(model, feature_cols, model_name, X_test, y_test):
    print("─" * 60)
    print("Feature Importances (Top 20)")
    print("─" * 60)

    if hasattr(model, "feature_importances_"):
        imp = model.feature_importances_
    elif hasattr(model, "named_steps"):
        # Pipeline (Logistic Regression)
        clf = model.named_steps["clf"]
        if hasattr(clf, "coef_"):
            imp = np.abs(clf.coef_[0])
        else:
            imp = np.zeros(len(feature_cols))
    else:
        imp = np.zeros(len(feature_cols))

    fi_df = pd.DataFrame({
        "feature": feature_cols,
        "importance": imp
    }).sort_values("importance", ascending=False).reset_index(drop=True)

    for _, row in fi_df.head(20).iterrows():
        bar = "█" * int(row["importance"] / fi_df["importance"].max() * 30)
        print(f"  {row['feature']:<35s}  {bar:<30s}  {row['importance']:.5f}")

    print()
    return fi_df

# 8. PREDICTION FUNCTION (for inference on new games)

def predict_game(model, feature_cols, game_dict):

#Predict the outcome of a single game.

    row = pd.DataFrame([game_dict])[feature_cols]
    prob = model.predict_proba(row)[0][1]
    pred = int(prob >= 0.50)

    if prob >= 0.65 or prob <= 0.35:
        confidence = "High"
    elif prob >= 0.55 or prob <= 0.45:
        confidence = "Medium"
    else:
        confidence = "Low"

    return {
        "home_win_probability":  round(float(prob), 4),
        "away_win_probability":  round(float(1 - prob), 4),
        "predicted_home_wins":   pred,
        "confidence":            confidence,
    }


# 9. SAVE PREDICTIONS ON TEST SET


def save_predictions(best_model, X_test, y_test, test_df, feature_cols):
    y_prob = best_model.predict_proba(X_test)[:, 1]
    y_pred = (y_prob >= 0.50).astype(int)

    pred_df = test_df[
        ["game_id","season","game_date","home_team","away_team","ballpark",
         "weather_condition","temperature_f","home_sp_name","away_sp_name",
         "home_team_win_pct","away_team_win_pct","home_runs_scored","away_runs_scored",
         "home_team_wins"]
    ].copy()

    pred_df["predicted_home_win_prob"] = y_prob.round(4)
    pred_df["predicted_home_wins"]     = y_pred
    pred_df["correct"]                 = (y_pred == y_test.values).astype(int)
    pred_df["winner_actual"]           = pred_df["home_team_wins"].map(
        {1: pred_df["home_team"], 0: pred_df["away_team"]}
    )
    # Vectorised winner mapping
    pred_df["winner_actual"] = np.where(
        pred_df["home_team_wins"] == 1,
        pred_df["home_team"],
        pred_df["away_team"]
    )
    pred_df["winner_predicted"] = np.where(
        pred_df["predicted_home_wins"] == 1,
        pred_df["home_team"],
        pred_df["away_team"]
    )

    pred_df.to_csv("/Users/drewrasmussen/Desktop/Personal/DCL/dcl_predictions_season3.csv", index=False)
    print(f"Saved {len(pred_df)} game predictions → dcl_predictions_season3.csv\n")
    return pred_df
